In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
 
silver_path = "abfss://silver@netflixxdata.dfs.core.windows.net/"

In [0]:
@dp.materialized_view(name="gold_netflix_directors")
def gold_netflix_directors():
    df = spark.read.format("delta").load(
        silver_path + "netflix_directors"
    )
    return df

In [0]:
lookup_rules = {
    "valid_show_id": "show_id IS NOT NULL"
}


In [0]:
# Gold Netflix Cast
@dp.materialized_view(name="gold_netflix_cast")
@dp.expect_all_or_drop(lookup_rules)
def gold_netflix_cast():
    return spark.read.format("delta").load(
        silver_path + "netflix_cast"
    )
 
 
# Gold Netflix Category
@dp.materialized_view(name="gold_netflix_category")
@dp.expect_all_or_drop(lookup_rules)
def gold_netflix_category():
    return spark.read.format("delta").load(
        silver_path + "netflix_category"
    )
 
 
# Gold Netflix Countries
@dp.materialized_view(name="gold_netflix_countries")
@dp.expect_all_or_drop(lookup_rules)
def gold_netflix_countries():
    return spark.read.format("delta").load(
        silver_path + "netflix_countries"
    )

In [0]:
@dp.temporary_view(name="staging_netflix_titles")
def staging_netflix_titles():
    df = spark.read.format("delta").load(
        silver_path + "netflix_titles"
    )
    return df

In [0]:
@dp.temporary_view(name="transformed_netflix_titles")
def transformed_netflix_titles():
    df = spark.read.table("staging_netflix_titles")
 
    df = df.withColumn("new_flag", F.lit(1))
 
    return df

In [0]:
# Data quality rules for Netflix Titles
master_rules = {
    "valid_show_id": "show_id IS NOT NULL",
    "valid_new_flag": "new_flag IS NOT NULL"
}
 
@dp.materialized_view(name="gold_netflix_titles")
@dp.expect_all_or_drop(master_rules)
def gold_netflix_titles():
    df = spark.read.table("transformed_netflix_titles")
    return df